# 01 — Cleaning WITHOUT removing duplicate rows

Reproduces the exact cleaning pipeline from `notebooks/01_Data_Cleaning.ipynb`, **except for the
duplicate-row removal step**.

**Isolation:** this notebook only **reads** `../../data/raw/` and `../../data/cleaned/`
(for the control check). It only **writes** to `experiment_nodedup/`. The base project's
`notebooks/`, `data/cleaned/`, `models/`, `reports/` directories are left untouched.

| Variant | Remove dupes | Remove conflicts |
|---|---|---|
| `V1_nodedup` | **no** | yes |
| `V2_nodedup_nocontra` | **no** | **no** |
| `V0_official_recipe` | yes | yes (control) |

`V0` must reproduce the official cleaned set value-for-value — otherwise,
every comparison afterward loses its control basis.

In [1]:
import json, os
import numpy as np
import pandas as pd
from fastai.tabular.all import df_shrink

# the notebook lives in experiment_nodedup/notebooks -> the base project is the grandparent directory
BASE = os.path.abspath(os.path.join('..', '..'))
NEW = os.path.join(BASE, 'experiment_nodedup')
RAW = os.path.join(BASE, 'data', 'raw', 'CSV Files', 'Training and Testing Sets')
OUTD = os.path.join(NEW, 'data', 'cleaned_nodedup')
OUTR = os.path.join(NEW, 'reports')
for p in [OUTD, OUTR, os.path.join(NEW, 'models')]:
    os.makedirs(p, exist_ok=True)
print('base project:', BASE)
print('writing to  :', NEW)

C:\venvs\ids5g\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


base project: G:\.shortcut-targets-by-id\1C2Fd9KvthwrCJvr4mxV0l6w7K1lsUbNE\5G_IDS_Project
writing to  : G:\.shortcut-targets-by-id\1C2Fd9KvthwrCJvr4mxV0l6w7K1lsUbNE\5G_IDS_Project\experiment_nodedup


In [2]:
CONTAMINATING = ['ct_state_ttl', 'sttl', 'dttl', 'ct_dst_ltm', 'ct_srv_dst',
                 'ct_dst_src_ltm', 'ct_srv_ltm', 'ct_srv_src', 'ct_dst_sport_ltm']
USELESS = ['is_sm_ips_ports', 'ct_ftp_cmd', 'is_ftp_login']
TARGETS = ['label', 'attack_cat']


def clean(csv_path, drop_dups, drop_contradictions):
    """Same as notebook 01's clean() function; the two row-removal steps are toggled by flags."""
    df = pd.read_csv(csv_path, skipinitialspace=True, low_memory=False, encoding='utf-8')
    st = {'rows_raw': len(df), 'attack_rate_raw': float(df['label'].mean())}
    df.drop(columns=['id'], inplace=True)
    df = df_shrink(df, skip=[], obj2cat=True, int2uint=False)
    st['nan'] = int(df.isna().sum().sum())
    to_drop = CONTAMINATING + USELESS
    st['not_in_data'] = [c for c in to_drop if c not in df.columns]
    df.drop(columns=[c for c in to_drop if c in df.columns], inplace=True)

    feat = [c for c in df.columns if c not in TARGETS]
    st['duplicate_rows_present'] = int(df.duplicated().sum())
    if drop_dups:
        n = len(df); df = df.drop_duplicates(); st['duplicates_removed'] = n - len(df)
    else:
        st['duplicates_removed'] = 0

    n_lab = df.groupby(feat, observed=True, dropna=False)['label'].transform('nunique')
    st['contradictory_present'] = int((n_lab > 1).sum())
    if drop_contradictions:
        df = df[n_lab == 1]; st['contradictory_removed'] = st['contradictory_present']
    else:
        st['contradictory_removed'] = 0

    df = df.reset_index(drop=True)
    st['rows_final'], st['cols_final'] = df.shape
    st['attack_rate_final'] = float(df['label'].mean())
    return df, st

In [3]:
variants = {
    'V1_nodedup':          dict(drop_dups=False, drop_contradictions=True),
    'V2_nodedup_nocontra': dict(drop_dups=False, drop_contradictions=False),
    'V0_official_recipe':  dict(drop_dups=True,  drop_contradictions=True),
}
stats, frames = {}, {}
for vname, kw in variants.items():
    stats[vname] = {}
    for split in ['training', 'testing']:
        df, st = clean(os.path.join(RAW, f'UNSW_NB15_{split}-set.csv'), **kw)
        stats[vname][split] = st; frames[(vname, split)] = df
        print(f"[{vname}/{split}] {st['rows_raw']:,} -> {st['rows_final']:,} rows | "
              f"Attack {st['attack_rate_raw']:.4f} -> {st['attack_rate_final']:.4f} | "
              f"dupes: present {st['duplicate_rows_present']:,} removed {st['duplicates_removed']:,} | "
              f"conflicts: present {st['contradictory_present']:,} removed {st['contradictory_removed']:,}")

[V1_nodedup/training] 175,341 -> 170,515 rows | Attack 0.6806 -> 0.6807 | dupes: present 75,775 removed 0 | conflicts: present 4,826 removed 4,826


[V1_nodedup/testing] 82,332 -> 80,816 rows | Attack 0.5506 -> 0.5560 | dupes: present 31,523 removed 0 | conflicts: present 1,516 removed 1,516


[V2_nodedup_nocontra/training] 175,341 -> 175,341 rows | Attack 0.6806 -> 0.6806 | dupes: present 75,775 removed 0 | conflicts: present 4,826 removed 0


[V2_nodedup_nocontra/testing] 82,332 -> 82,332 rows | Attack 0.5506 -> 0.5506 | dupes: present 31,523 removed 0 | conflicts: present 1,516 removed 0


[V0_official_recipe/training] 175,341 -> 98,934 rows | Attack 0.6806 -> 0.4964 | dupes: present 75,775 removed 75,775 | conflicts: present 632 removed 632


[V0_official_recipe/testing] 82,332 -> 50,567 rows | Attack 0.5506 -> 0.3665 | dupes: present 31,523 removed 31,523 | conflicts: present 242 removed 242


### Required control check

The `V0_official_recipe` variant must reproduce `data/cleaned` **value-for-value**.

In [4]:
for split in ['training', 'testing']:
    official = pd.read_parquet(os.path.join(BASE, 'data', 'cleaned', f'UNSW_NB15_{split}-set.parquet'))
    mine = frames[('V0_official_recipe', split)][list(official.columns)].reset_index(drop=True)
    same = mine.equals(official.reset_index(drop=True))
    print(f'[{split}] V0 == data/cleaned parquet -> {same}')
    assert same, 'The cleaning function failed to reproduce the official set'
print('\nThe cleaning function is a faithful copy. Every comparison afterward has a valid control.')

[training] V0 == data/cleaned parquet -> True
[testing] V0 == data/cleaned parquet -> True

The cleaning function is a faithful copy. Every comparison afterward has a valid control.


## Does the cleaning step create the class-ratio skew?

The earlier report claims the cleaning step **by itself** creates a 13-point skew between train and test.
Here we measure it directly: compare the gap **before** and **after** cleaning.

In [5]:
gap = pd.DataFrame([{
    'variant': v,
    'train_raw': stats[v]['training']['attack_rate_raw'],
    'test_raw': stats[v]['testing']['attack_rate_raw'],
    'gap_raw': stats[v]['training']['attack_rate_raw'] - stats[v]['testing']['attack_rate_raw'],
    'train_clean': stats[v]['training']['attack_rate_final'],
    'test_clean': stats[v]['testing']['attack_rate_final'],
    'gap_clean': stats[v]['training']['attack_rate_final'] - stats[v]['testing']['attack_rate_final'],
} for v in variants]).set_index('variant')
display(gap.round(4))

v0 = stats['V0_official_recipe']
print(f"The official cleaning pulls the train Attack ratio down by "
      f"{v0['training']['attack_rate_raw'] - v0['training']['attack_rate_final']:.4f} points")
print(f"                      and the test  ratio down by "
      f"{v0['testing']['attack_rate_raw'] - v0['testing']['attack_rate_final']:.4f} points")
print('=> Both sets are pulled down by almost the SAME amount, so the gap stays unchanged.')

,train_raw,test_raw,gap_raw,train_clean,test_clean,gap_clean
variant,,,,,,
V1_nodedup,0.6806,0.5506,0.13,0.6807,0.5560,0.1247
V2_nodedup_nocontra,0.6806,0.5506,0.13,0.6806,0.5506,0.1300
V0_official_recipe,0.6806,0.5506,0.13,0.4964,0.3665,0.1299


The official cleaning pulls the train Attack ratio down by 0.1842 points
                      and the test  ratio down by 0.1841 points
=> Both sets are pulled down by almost the SAME amount, so the gap stays unchanged.


## The cost of keeping duplicate rows: test leaking into train

In [6]:
cross = {}
for vname in variants:
    tr, te = frames[(vname, 'training')], frames[(vname, 'testing')]
    key = [c for c in tr.columns if c not in TARGETS] + ['label']
    tr_keys = set(tr[key].astype(str).agg('|'.join, axis=1))
    hit = te[key].astype(str).agg('|'.join, axis=1).isin(tr_keys)
    cross[vname] = {'test_rows': len(te), 'overlap_with_train': int(hit.sum()), 'rate': float(hit.mean())}
display(pd.DataFrame(cross).T.assign(**{'rate': lambda d: d['rate'].map('{:.2%}'.format)}))

,test_rows,overlap_with_train,rate
V1_nodedup,80816.0,20960.0,25.94%
V2_nodedup_nocontra,82332.0,23536.0,28.59%
V0_official_recipe,50567.0,1993.0,3.94%


In [7]:
for vname in ['V1_nodedup', 'V2_nodedup_nocontra']:
    for split in ['training', 'testing']:
        p = os.path.join(OUTD, f'{vname}_UNSW_NB15_{split}-set.parquet')
        frames[(vname, split)].to_parquet(p)
        print('saved', os.path.relpath(p, BASE))

with open(os.path.join(OUTR, 'cleaning_stats.json'), 'w', encoding='utf-8') as f:
    json.dump({'stats': stats, 'prevalence_gap': gap.round(6).to_dict('index'),
               'cross_set_duplicates': cross}, f, ensure_ascii=False, indent=2)
print('saved reports/cleaning_stats.json')

saved experiment_nodedup\data\cleaned_nodedup\V1_nodedup_UNSW_NB15_training-set.parquet
saved experiment_nodedup\data\cleaned_nodedup\V1_nodedup_UNSW_NB15_testing-set.parquet


saved experiment_nodedup\data\cleaned_nodedup\V2_nodedup_nocontra_UNSW_NB15_training-set.parquet
saved experiment_nodedup\data\cleaned_nodedup\V2_nodedup_nocontra_UNSW_NB15_testing-set.parquet
saved reports/cleaning_stats.json
